## Overview

In this assignment we will be predicting stroke. 
There are three learning outcomes: 

  1. To be able to consider relevant metrics for a given prediction task

  2. To evaluate trade-offs in the data inclusion process
    
  3. To be able to read existing python files and understanding their functionality

This notebook will focus on the second outcome. 

Generally, in machine learning tasks we use all available data to train a model. However, when there are subpopulations who may differ from the general population these population level models may underperform for those groups. Here we are going to explore different settings of training and testing to evaluate performance for particular groups. 

You will be adding functions to the ```find_best_hyperparameters.py``` file. 

In each setting we want to look at group-level average precision as well as population level average precision. The difference will come from the dataset we use to pick the best hyper-parameters, as well as whether or not we constrain our model with a fairness constraint. 



## Pipeline task overview: Predicting stroke

Recall from your previous courses that these tasks can typically be described by the following components: 

 1. Data collection - <font color='green'>Done</font>
 2. Data cleaning / transformation - <font color='green'>Done</font>
 3. Dataset splitting <font color='magenta'>You will do in notebook 1b </font>
 4. Model training <font color='magenta'>You will do</font>
 5. Model evaluation <font color='magenta'>You will do</font>
 6. Repeat 1-5 to perform model selection <font color='magenta'>You will do</font>
 7. Presenation of findings (Visualization) <font color='green'>Not required</font>

![Flowchart of Grid Search Workflow](assignment1/training.png)

Source: [Scikit-Learn's Cross-validation documentation](https://scikit-learn.org/stable/modules/cross_validation.html)

### Setting one - picking hyperparameters using population level data 


Here, we want to use all of the available data to pick the best model parameters - exactly like we need in assignment 1 a, the difference is that after finding the best hyper-parameters we want to evaluate the model one group at a time.

We've written some functions to help you with this task. 

In [1]:
import warnings
warnings.simplefilter("ignore")

import sys
import os
from mads.lib.path import assets

data_cleaners = assets.find("shared_files/assignment1/data_cleaners.py")
sys.path.append(os.path.dirname(data_cleaners))
import data_cleaners as dc

In [2]:
stroke_data = dc.StrokeKaggle(assignment_1a=False)

In [3]:
group_train, pop_cv = stroke_data.get_all_train_test_indices_group() 

In [4]:
group_train.keys()


dict_keys(['80 and over', 'aged', 'middle aged'])

In the last notebook we used a dictionary where the keys were fold ids and the values were tuples contained training indices and testing indices. The dictionary `group_train` is similar in structure, it just has one extra layer of complexity. Here, each key is an age category. The values are then dictionaries just like in the last notebook, the keys are fold ids and the values are tuples containing training indices and testing indices. 

The object ``pop_cv``, is the same cross fold lookup as from the last notebook. 

You will complete the function ```run_gridsearch_all_evaluate_group(group_cv,pop_cv,dataset,metric,threshold)``` 

passing in `pop_cv` is not strictly necessary as it contains the same information as `group_train`, but it will allow you to more easily access the population level training data. For example, if you want to get the training indices for the entire population in fold one, you can just call `pop_cv[0][0]`, whereas if you want to evaluate a model only on individuals age 80 and above, you can access this data using group train `group_train[0][1]`. 

In [5]:
find_best_hyperparameters = assets.find("shared_files/assignment1/find_best_hyperparameters.py")
sys.path.append(os.path.dirname(find_best_hyperparameters))
import find_best_hyperparameters as fbh

In [6]:
%load_ext autoreload
%autoreload 2

## <font color='magenta'>Task Zero</font>

Use Pandas groupby to determine, for each category in the column `age_cat` the number of people who did and did not have a stroke. To receive full credit you must use `groupby`. 

In [ ]:
def get_counts_age_category(stroke_data):
    # group records by age category and stroke outcome
    # 0 = no stroke, 1 = stroke

    # count observations to examine class distribution across age groups
    ser = stroke_data.total_dataset.groupby(
        ["age_cat", "stroke"]).size()
    
    return ser

# display stroke counts for each age category
get_counts_age_category(stroke_data)

age_cat      stroke
80 and over  0          146
             1           40
aged         0          722
             1          119
middle aged  0         1470
             1           80
dtype: int64

In [8]:
#hidden tests are within this cell

## <font color='magenta'>Task One</font>


You will complete the function ```run_gridsearch_all_evaluate_group```  in ``` find_best_hyperparameters```. This function is part of a larger code base. A good first step before getting started is to make sure you understand what each existing function in this codebase does. For example, the function ```run_gridsearch_all_evaluate_all``` will be really helpful when implementing ```run_gridsearch_all_evaluate_group```. To get started, read through ```run_gridsearch_all_evaluate_all``` and think about how you will need to change ```run_gridsearch_all_evaluate_group``` to complete the task(s) explained below. 

In this task, you will add the missing code to 

```run_gridsearch_all_evaluate_group```  in ``` find_best_hyperparameters```

You will see that this function takes in ```group_specific_cv``` as an argument. This function will find the best hyper-parameters on the whole dataset, and then for those best hyper-parameters it will return the average precision for the group whose test data is stored in ```group_specific_cv```. For example, look at the code below, 

```python 
    for k,v in group_train.items():
        result[k] = fbh.run_gridsearch_all_evaluate_group(v,pop_cv,stroke_data,'average_precision',None)
```
For each group in group_train.items() you will pass in that group's training data `v` to the function ```run_gridsearch_all_evaluate_group```.

Let's say `k = aged` ```run_gridsearch_all_evaluate_group``` does two things:

 1. It finds the best hyper-parameters using `pop_cv` and `stroke_data`, exactly as ```fbh.run_gridsearch_all_evaluate_all```. For example, let's say it finds that the best penalty is None, and the best value of C is .01. 
 2. For `penalty = none` and `C = .01` train the logistic regression on the whole population training data `pop_cv` of each fold, but calculate the average precision for the `aged` group only. 

It returns (best_params, average score) for each group. For example, result should look like the following (note the numbers below are just examples):

```python 
{'80 and over': ('l2-0.1', 0.41925690302157843),
 'aged': ('l2-0.1', 0.21005690472838787),
 'middle aged': ('l2-0.1', 0.51908823330132199)}

```

In [9]:
result = {}
# evaluate the population-trained model for each age group
for k, v in group_train.items():

    result[k] = fbh.run_gridsearch_all_evaluate_group(
        v, pop_cv, stroke_data, 'average_precision', None
    )

    print(k, result[k])

80 and over ('l2-1.0', 0.2951992811550253)
aged ('l2-1.0', 0.27048954447872536)
middle aged ('l2-1.0', 0.1028388633010957)


In [10]:
#hidden tests are within this cell

## <font color='magenta'>Task Two</font>


In this task you will implement

```run_gridsearch_on_group_only```  in ``` find_best_hyperparameters```. 

If you inspect the following code:

```python 
    for k,v in group_train.items():
        result[k] = fbh.run_gridsearch_on_group_only(v,stroke_data,'average_precision',None)
```
you see that this function takes only the training data for a single group. In then does two things: 

 1. It finds the best hyper-parameters using `v` and `stroke_data`, as if you only had access to data for the group `k`. For example, let's say it finds that the best penalty is None, and the best value of C is .01. 
 2. For `penalty = none` and `C = .01` train the logistic regression on all of k's training data, and calculate the average precision for the `k` group only. 

It returns (best_params, average score) for each group. Like above, result should have the following form (note the numbers below are simple examples):

```python 
{'80 and over': ('l2-0.1', 0.41925690302157843),
 'aged': ('l2-0.01', 0.21005690472838787),
 'middle aged': ('l2-0.001', 0.51908823330132199)}

```

In [ ]:
# collect population-trained model scores for each age group
result = {}
# train and evaluate separate models for each age group
for k, v in group_train.items():

    try:
        # fit on population folds and evaluate the selected age group's validation data
        result[k] = fbh.run_gridsearch_on_group_only(
            v, stroke_data, 'average_precision', None
        )

        print(k, result[k])

    except Exception as e:
        print(e)
        print("no 0s")

80 and over ('l2-0.01', 0.22600778444614034)
aged ('l2-10.0', 0.27360273717067873)
middle aged ('l2-0.001', 0.1311401974221944)


In [12]:
#hidden tests are within this cell

## <font color='magenta'>Task Three A (Optional)</font>


In this task you will use customized weights when you train the Logistic Regression model. The first thing you need to do is to create a dictionary of the following structure:

```python
{
'middle aged' :
    {
        1: weight_middle_aged_1,
        0: weight_middle_aged_0
    },
'aged' :
    {
        1: weight_aged_1,
        0: weight_aged_0
    },
'80 and over' :
    {
        1: weight_80_1,
        0: weight_80_0
    }
}
```

Where each weight in this dictionary is calculated according to the equation shown in the lecture. 

You will set the variable ```weights_dict``` in the following cell by filling in the method `instantiate_weights_dict in find_best_hyperparameters.py` which takes as input `stroke_data.total_dataset` and returns a dictionary in the form above. 

In [ ]:
# instantiate the weights dictionary over the whole dataset using age_num and stroke columns
# complete the code in instantiate_weights_dict in find_best_hyperparameters.py

# create class weights for each age group from the complete dataset
# requires implementing `instantiate_weights_dict` in find_best_hyperparameters.py
weights_dict = fbh.instantiate_weights_dict(stroke_data.total_dataset)
weights_dict

{'middle aged': {0: 0.9566310031967774, 1: 1.7969053162592161},
 'aged': {0: 1.056790465840479, 1: 0.6554393585140692},
 '80 and over': {0: 1.1558199244103529, 1: 0.43125727590221186}}

In [14]:
#hidden tests are within this cell

## <font color='magenta'>Task Three B (Optional)</font>



In this task you will implement

```get_best_hyperparameters_weighted```  in ```find_best_hyperparameters```. 


This function will return the best values for `C` and `penalty`. Make sure you note these values as you will use them in Task Three C. 

Here you are using all of the data to train and test. All of the group specific information is passed in through the weights dict. 

In [ ]:
# tune the weighted logistic regression on population-wide cross-validation folds
# requires implementing `get_best_hyperparameters_weighted` in the helper file

# choose hyperparameters using weighted population training folds
C, penalty = fbh.get_best_hyperparameters_weighted(
    pop_cv,stroke_data,weights_dict
)

print('Best C:', C, '| Best penalty', penalty)

Best C: 0.01 | Best penalty None


## <font color='magenta'>Task Three C (Optional)</font>



In this task you will implement

```run_gridsearch_all_evaluate_group_resampled```  in ```find_best_hyperparameters```. 

If you inspect the following code in the cell below:

```python 
    for k,v in group_train.items():
        result[k] = fbh.run_gridsearch_all_evaluate_group_resampled(v,pop_cv,stroke_data,weights_dict, C, penalty)
```
you see that this function takes both the training data for one group (`v`), and for all everyone (`pop_cv`).

Each time you call `run_gridsearch_all_evaluate_group_resampled` you will do the following:
    
For each fold - call `get_predictions_weights`. The most important thing to keep track of here is that 
    `weights_train`, `X_train`, and `y_train` depend on all of the data in `pop_cv`. On the other hand,
    `weights_test`, `X_test`, and `y_test` depend on the data for this group. 
`Return` - the average precision across all k folds for this group. For example:

```python 
{'80 and over':  0.41925690302157843,
 'aged': 0.21005690472838787,
 'middle aged':  0.51908823330132199}

```


In [ ]:
result = {}
# compare weighted population-model performance across age groups
for k, v in group_train.items():

    try:
        # train on weighted population data and score age-group validation data
        result[k] = fbh.run_gridsearch_all_evaluate_group_resampled(
            v, pop_cv, stroke_data, weights_dict, C, penalty
        )

        print(k, result[k])

    except Exception as e:
        print(e)
        print("no 0s")

80 and over 0.32114445920512813
aged 0.2402512829635417
middle aged 0.11780488283418648


In [17]:
#hidden tests are within this cell

## <font color='magenta'>Task Four</font>



In this task you will implement

```run_gridsearch_on_all_constrained_with_model```  in ```find_best_hyperparameters```. 

If you inspect the following code:

```python 
    for k,v in group_train.items():
        result[k] = fbh.run_gridsearch_on_all_constrained_with_model(v,pop_cv,stroke_data)
```
you see that this function takes both the training data for one group (`v`), and for all everyone (`pop_cv`).It also requires `stroke_data`. 


In then does two things: 

 1. It calls `get_constrained_models` which returns a dictionary of trained models for each fold. *Note: there is no hyper-parameter search here*.
 
 2. Using this trained model predicts on the test set for group k and and calculates the average precision for the group `k` only. 
 
Unlike above, here we are just saving the best results, not the hyper-parameters. Result should take the following form:

```python 
{'80 and over':  0.41925690302157843,
 'aged': 0.21005690472838787,
 'middle aged': 0.51908823330132199}

```

In [ ]:
# store age-group results for fairness-constrained models
result = {}
# evaluate constrained models for each age group
for k, v in group_train.items():

    try:
        # evaluate each constrained population model on the age group's validation data
        result[k] = fbh.run_gridsearch_on_all_constrained_with_model(
            v, pop_cv, stroke_data
        )

        print(k, result[k])

    except Exception as e:
        print(e)
        print("no 0s")

80 and over 0.2511773800429263
aged 0.20238548512330076
middle aged 0.07459073939241706


In [19]:
#hidden tests are within this cell

## IMPORTANT: Uncomment the following code when you are ready to move on to the next assignment

In [ ]:
script = assets.find('copy_shared_files.sh')
!bash {script}

=== Copying shared files back to coursework ===
